# AI Lab - Practical Question No. 29
## A* Pathfinding in 2D Maze (Informed Search & Spatial Navigation)

| Specification | Detail |
| :--- | :--- |
| **Title** | A* Pathfinding in 2D Maze |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🟠 Advanced |
| **Practical Type** | Simulation |
| **Core AI Concept** | A* Application to spatial navigation, Informed Heuristic Search |
| **Evaluation Function** | $f(n) = g(n) + \text{manhattan}(n, \text{goal})$ |
| **Python Concepts Tested** | ASCII matrix manipulation, A* logic, Min-heap via `heapq` |
| **Complexity** | Time: Depends on heuristic accuracy; Space: $\mathcal{O}(V)$ |
| **Sample Input** | 5x5 Grid with obstacles |
| **Sample Output** | ASCII Grid output with `'X'`, `'.'`, `'*'`, and `'#'` |
| **Viva Topics** | • A* in video games<br>• Explored set size relative to BFS |
| **Optional Challenge** | Animate the expansion process in the terminal |

## 1. Theoretical Foundations & Evaluation Function

### Core Evaluation Function: $f(n) = g(n) + h(n)$
In A* pathfinding, states are evaluated using both past accumulated cost and estimated future cost:
$$f(n) = g(n) + h(n)$$
- **$g(n)$**: The exact accumulated step cost incurred from the start node to node $n$. On an unweighted 4-connected grid with uniform step cost $c = 1$, $g(n)$ equals the number of grid transitions.
- **$h(n)$**: The heuristic estimated distance from node $n$ to the goal coordinate $(r_{\text{goal}}, c_{\text{goal}})$. For orthogonal 4-directional motion (Up, Down, Left, Right), the canonical heuristic is the **Manhattan Distance** ($L_1$ norm):
  $$h_M((r, c), (r_{\text{goal}}, c_{\text{goal}})) = |r - r_{\text{goal}}| + |c - c_{\text{goal}}|$$
- **$f(n)$**: The estimated total cost of the cheapest complete path going from start to goal through node $n$.

### Admissibility and Consistency on Orthogonal Grids
1. **Admissibility ($0 \le h(n) \le h^*(n)$)**:
   In an obstacle-free grid, the Manhattan distance is the exact minimum number of orthogonal steps required to reach the target. Static obstacles can only block passages and force detours, meaning $h^*(n) \ge h_M(n)$ always. Manhattan distance is strictly admissible.
2. **Consistency / Monotonicity ($h(u) \le c(u, v) + h(v)$)**:
   For any two adjacent cells $u$ and $v$, $|h(u) - h(v)| \le 1 = c(u, v)$. This reverse triangle inequality ensures that $f(n)$ is monotonically non-decreasing along any path, guaranteeing that the first time a state is expanded from the priority queue, its $g(n)$ cost is strictly optimal without ever needing to reopen closed nodes.

### Output ASCII Glyph Specification
- `'X'`: Impassable obstacle / wall cell
- `'.'`: Open walkable cell (unvisited)
- `'*'`: Final optimal solution path
- `'#'`: Explored nodes expanded during search (not part of the final path)

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path for interactive imports
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.a_star_maze_solver import (
    DEFAULT_5X5_MAZE,
    DEFAULT_5X5_START,
    DEFAULT_5X5_GOAL,
    DEFAULT_10X10_MAZE,
    CONCAVE_TRAP_MAZE,
    SERPENTINE_5X5_MAZE,
    OPEN_5X5_MAZE,
    a_star_maze_solver,
    breadth_first_search_maze,
    compare_astar_vs_bfs,
    generate_expansion_frames,
    manhattan_distance,
    parse_grid,
    render_side_by_side_comparison,
    render_solved_maze,
    VIVA_VOCE_GUIDE,
)

print("A* 2D Maze Pathfinding modules and benchmarks loaded successfully!")

## 2. Canonical 5x5 Syllabus Sample Execution

Let us run A* Pathfinding on the 5x5 benchmark maze from Start `(0, 0)` to Goal `(4, 4)`.
The 5x5 grid contains a 3-cell vertical obstacle wall along column 2 blocking direct horizontal movement.
We verify that the solved maze is rendered with `'X'` for obstacles, `'.'` for open, `'*'` for path, and `'#'` for explored nodes.

In [ ]:
result = a_star_maze_solver(
    maze=DEFAULT_5X5_MAZE,
    start=DEFAULT_5X5_START,
    goal=DEFAULT_5X5_GOAL,
    tie_breaker="fifo",
    record_steps=True,
)

print("=== A* 2D Maze Pathfinding Result ===")
print(f"Goal Found      : {result.found}")
print(f"Optimal Steps   : {result.path_length} transitions ({len(result.path)} coordinates)")
print(f"Nodes Expanded  : {result.nodes_expanded} nodes")
print(f"Explored Set    : {len(result.explored_set)} cells")
print(f"Execution Time  : {result.execution_time_ms:.3f} ms")
print("\n--- ASCII Grid Visualization (Exact Syllabus Output) ---")
rendered_grid = render_solved_maze(
    maze=DEFAULT_5X5_MAZE,
    path=result.path,
    explored=result.explored_set,
    preserve_endpoints=False,
    show_coordinates=False,
)
print(rendered_grid)

## 3. Step-by-Step Priority Queue Trace Table

Let us inspect the chronological expansion steps, verifying how $g(n)$, $h(n)$, and $f(n)$ guide the priority queue.

In [ ]:
print(f"{'Step':<5} | {'Node (r,c)':<10} | {'g(n)':<5} | {'h(n)':<5} | {'f(n)':<5} | {'Action':<14} | {'Queue':<6} | {'Evaluated / Enqueued'}")
print("-" * 86)
for st in result.steps:
    eval_str = f"eval={len(st.neighbors_evaluated)}, push={len(st.neighbors_enqueued)}"
    print(
        f"{st.step_number:<5} | {str(st.current_coord):<10} | "
        f"{st.g_cost:<5} | {st.h_cost:<5} | {st.f_cost:<5} | "
        f"{st.action:<14} | {st.frontier_size:<6} | {eval_str}"
    )
print("-" * 86)
print(f"Path Trajectory: {' -> '.join(str(pt) for pt in result.path)}")

## 4. 2D Grid Parser Flexibility

The module provides a robust parser `parse_grid` that accepts:
- Multiline ASCII strings with space-separated tokens or contiguous characters
- 2D lists of characters (`'X'`, `'.'`, `'S'`, `'G'`)
- 2D lists of integers (`0` = open, `1` = obstacle)
- Automatic detection of Start (`'S'`) and Goal (`'G'`) coordinates

In [ ]:
custom_ascii = """
S . . X .
. X . X .
. X . . .
. X X X .
. . . . G
"""

custom_maze, start_pt, goal_pt = parse_grid(custom_ascii)
custom_res = a_star_maze_solver(custom_maze, start=start_pt, goal=goal_pt)

print(f"Parsed Grid Dimensions : {len(custom_maze)}x{len(custom_maze[0])}")
print(f"Detected Start / Goal  : {start_pt} -> {goal_pt}")
print(f"Optimal Path Length    : {custom_res.path_length} steps")
print(f"Nodes Expanded         : {custom_res.nodes_expanded} nodes")
print("\nRendered Solved Grid (Preserving 'S' and 'G'):")
print(render_solved_maze(custom_maze, custom_res.path, custom_res.explored_set, preserve_endpoints=True, show_coordinates=True))

## 5. Comparative Performance Analysis: A* vs. BFS vs. Greedy

A fundamental theme in university AI curricula is contrasting **Informed A* Search** against **Blind Breadth-First Search (BFS)** and **Pure Greedy Best-First Search (GBFS)**:
- **Exploration Efficiency**: A* focuses the search wavefront toward the goal using $h(n)$, cutting explored set sizes significantly compared to BFS.
- **Solution Optimality**: While Greedy BFS often takes suboptimal detours, A* is mathematically guaranteed to find the strictly optimal shortest path.

In [ ]:
# Benchmark comparison on 10x10 maze
comp_10x10 = compare_astar_vs_bfs(DEFAULT_10X10_MAZE, (0, 0), (9, 9), include_gbfs=True)

print("=" * 72)
print("10x10 MAZE BENCHMARK: A* vs. BFS vs. GREEDY BEST-FIRST")
print("=" * 72)
print(f"{'Metric':<28} | {'A* Search':<16} | {'BFS (Blind)':<16} | {'Greedy BFS'}")
print("-" * 72)
print(f"{'Evaluation Function':<28} | {'f = g + h':<16} | {'f = depth':<16} | {'f = h'}")
print(f"{'Nodes Expanded':<28} | {f'{comp_10x10.astar_result.nodes_expanded} nodes':<16} | {f'{comp_10x10.bfs_result.nodes_expanded} nodes':<16} | {f'{comp_10x10.gbfs_result.nodes_expanded} nodes'}")
print(f"{'Path Length':<28} | {f'{comp_10x10.astar_result.path_length} steps':<16} | {f'{comp_10x10.bfs_result.path_length} steps':<16} | {f'{comp_10x10.gbfs_result.path_length} steps'}")
print(f"{'Path Optimality':<28} | {'OPTIMAL':<16} | {'OPTIMAL':<16} | {'SUBOPTIMAL (+4)'}")
print("=" * 72)
print("\nVisual Side-by-Side Comparison:")
print(render_side_by_side_comparison(comp_10x10))

## 6. Multi-Benchmark Evaluation Suite

Evaluating A* across diverse spatial topologies:
1. **Serpentine Corridor**: Forces winding paths through narrow bottlenecks.
2. **Concave Trap**: U-shaped obstacle testing escape from local heuristic minima.
3. **Open Grid**: Obstacle-free grid where path length strictly equals Manhattan distance.

In [ ]:
topologies = [
    ("Serpentine Corridor (5x5)", SERPENTINE_5X5_MAZE, (0, 0), (4, 4)),
    ("Concave Trap (10x10)", CONCAVE_TRAP_MAZE, (0, 0), (9, 9)),
    ("Open Obstacle-Free Grid (5x5)", OPEN_5X5_MAZE, (0, 0), (4, 4)),
]

for name, maze_grid, start, goal in topologies:
    res = a_star_maze_solver(maze_grid, start=start, goal=goal)
    print(f"\n[Topology] {name}")
    print(f"Optimal Steps: {res.path_length} | Nodes Expanded: {res.nodes_expanded} | Explored: {len(res.explored_set)}")
    print(render_solved_maze(maze_grid, res.path, res.explored_set))

## 7. Optional Challenge: Stepping Through Node Expansion Frames

The generator `generate_expansion_frames` yields snapshot states at each expansion step.
Let us inspect a sequence of frames showing how the explored set `'#'` advances and how the final path `'*'` emerges.

In [ ]:
frames = list(generate_expansion_frames(DEFAULT_5X5_MAZE, (0, 0), (4, 4)))
print(f"Total expansion frames generated: {len(frames)}\n")

# Display early frame, middle frame, and final frame
for frame_idx in [0, len(frames) // 2, len(frames) - 1]:
    f = frames[frame_idx]
    print(f"--- Frame #{f['frame_number']} | Current: {f['current']} | Action: {f['action']} | Explored: {f['explored_count']} ---")
    print(f['rendered_grid'])
    print()

## 8. University Viva Voce Preparation Guide

Comprehensive theoretical notes addressing key oral examination questions.

In [ ]:
for topic_title, content in VIVA_VOCE_GUIDE.items():
    print(f"### {topic_title.upper()}")
    print(f"Q: {content['question']}")
    print(f"A:\n{content['answer']}")
    print("-" * 78)